# API & MCP App Research Agent — Jupyter Notebook

This notebook creates a practical research pipeline for your Product Ops assignment.

**What it does**
1. Takes a list of apps and categories.
2. Searches the web for each app's official API/developer documentation.
3. Reads selected pages and extracts structured facts using an LLM.
4. Saves results and evidence URLs to CSV/JSON as it goes.
5. Optionally runs a second-pass verification on a sample.
6. Produces simple category/authentication/buildability summaries.

**Important:** the agent must not guess. If documentation does not confirm a field, it should return `Unknown` and flag it for manual review. Review the evidence before submitting your final report.

**Services used:** Tavily for web search and OpenAI for extraction. Composio is an integration/tool-execution platform; it is not a substitute for the web-search source. An optional Composio MCP connection example is included near the end.

## 1. Install dependencies

Run this cell once. If Jupyter asks you to restart the kernel after installation, do so and continue from the next cell.

In [ ]:
%pip install -q openai requests beautifulsoup4 pandas pydantic python-dotenv matplotlib

## 2. Add your API keys securely

Create accounts and obtain API keys from the providers. Enter keys when prompted; they will not be written into this notebook.

- OpenAI API key
- Tavily API key

Do not paste API keys into a public GitHub repository.

In [ ]:
import os
from getpass import getpass

OPENAI_API_KEY = getpass("OpenAI API key: ").strip()
TAVILY_API_KEY = getpass("Tavily API key: ").strip()

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

assert OPENAI_API_KEY and TAVILY_API_KEY, "Both API keys are required."
print("API keys loaded for this notebook session.")

## 3. Imports and configuration

In [ ]:
import json
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Optional
from urllib.parse import urlparse
from IPython.display import display

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

MODEL = "gpt-4.1-mini"
TAVILY_URL = "https://api.tavily.com/search"

# Keep this modest while testing. Increase after checking cost and output quality.
MAX_SEARCH_RESULTS = 5
MAX_PAGES_TO_READ = 3
REQUEST_TIMEOUT = 20
PAUSE_BETWEEN_APPS_SECONDS = 1.0

print("Ready.")

## 4. Prepare the app list

Replace the sample rows with the 100 apps from your assignment. Keep one row per app and use the categories from your assignment.

You can also load your own CSV. It must contain `app_name` and `category` columns.

In [ ]:
sample_apps = [
    {"app_name": "Slack", "category": "Communication"},
    {"app_name": "Microsoft Teams", "category": "Communication"},
    {"app_name": "Zoom", "category": "Communication"},
    {"app_name": "Gmail", "category": "Email"},
    {"app_name": "Mailchimp", "category": "Email"},
    {"app_name": "HubSpot", "category": "CRM"},
    {"app_name": "Salesforce", "category": "CRM"},
    {"app_name": "Stripe", "category": "Payments"},
    {"app_name": "PayPal", "category": "Payments"},
    {"app_name": "Asana", "category": "Project Management"},
]

apps_df = pd.DataFrame(sample_apps)
display(apps_df)

# OPTIONAL: load your own CSV instead:
# apps_df = pd.read_csv("apps.csv")
# Required columns: app_name, category

assert {"app_name", "category"}.issubset(apps_df.columns)
apps_df = apps_df.drop_duplicates(subset=["app_name"]).reset_index(drop=True)
print(f"Apps queued: {len(apps_df)}")

## 5. Web search tool

The agent searches for official developer/API documentation first. Search results are leads, not proof; the extraction prompt will require evidence URLs and will allow `Unknown`.

In [ ]:
def tavily_search(query: str, max_results: int = MAX_SEARCH_RESULTS) -> list:
    payload = {
        "api_key": os.environ["TAVILY_API_KEY"],
        "query": query,
        "search_depth": "advanced",
        "max_results": max_results,
        "include_answer": False,
        "include_raw_content": False,
    }
    response = requests.post(TAVILY_URL, json=payload, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    data = response.json()
    return data.get("results", [])


def search_app_docs(app_name: str) -> list:
    queries = [
        f"{app_name} official developer API documentation authentication",
        f"{app_name} official API getting started developer portal",
        f"{app_name} official MCP server Model Context Protocol",
    ]
    combined = []
    seen_urls = set()

    for query in queries:
        try:
            results = tavily_search(query)
            for item in results:
                url = item.get("url", "")
                if url and url not in seen_urls:
                    seen_urls.add(url)
                    combined.append({
                        "title": item.get("title", ""),
                        "url": url,
                        "content": item.get("content", ""),
                        "score": item.get("score", None),
                    })
        except Exception as e:
            print(f"Search warning for {app_name}: {e}")

    return combined


# Test the search tool on one app before running the full list.
test_results = search_app_docs(apps_df.iloc[0]["app_name"])
display(pd.DataFrame(test_results).head(8))

## 6. Read web pages

Some websites block automated requests or render content with JavaScript. This simple reader works for many static documentation pages; blocked/empty pages should be checked manually in a browser.

In [ ]:
def read_webpage(url: str) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; AppResearchAgent/1.0)"
    }
    try:
        response = requests.get(
            url, headers=headers, timeout=REQUEST_TIMEOUT
        )
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "nav", "footer", "header", "noscript"]):
            tag.decompose()

        text = soup.get_text(" ", strip=True)
        text = re.sub(r"\s+", " ", text)
        return text[:12000]
    except Exception as e:
        return f"[PAGE_READ_ERROR] {type(e).__name__}: {e}"


def collect_sources(app_name: str) -> list:
    search_results = search_app_docs(app_name)
    sources = []

    for result in search_results[:MAX_PAGES_TO_READ]:
        page_text = read_webpage(result["url"])
        sources.append({
            "title": result["title"],
            "url": result["url"],
            "search_snippet": result["content"],
            "page_text": page_text,
        })

    return sources


# Quick test
test_sources = collect_sources(apps_df.iloc[0]["app_name"])
print("Sources collected:", len(test_sources))
for s in test_sources:
    print("\nTITLE:", s["title"])
    print("URL:", s["url"])
    print("TEXT PREVIEW:", s["page_text"][:350])

## 7. Define the structured research record

These fields are the output schema. The agent must distinguish confirmed facts from missing information.

In [ ]:
class AppResearchRecord(BaseModel):
    app_name: str
    category: str
    description: str
    authentication_methods: List[str] = Field(default_factory=list)
    credential_access: str  # Self-serve, Gated, Mixed, Unknown
    api_available: str      # Yes, No, Unknown
    api_type_or_surface: str
    api_docs_url: str
    mcp_status: str          # Native MCP, Third-party/Composio integration, No evidence, Unknown
    buildability: str        # Easy, Moderate, Difficult, Unknown
    main_blocker: str
    evidence: List[str] = Field(default_factory=list)
    confidence: str          # High, Medium, Low
    needs_manual_review: bool
    notes: str = ""

## 8. Extraction agent

The LLM reads the search evidence and page text, then returns one JSON record. It is instructed not to infer unsupported claims.

For credential access:
- **Self-serve:** a developer can register and create credentials without contacting sales/support.
- **Gated:** approval, sales contact, partner status, or manual review is required.
- **Mixed:** some API access is self-serve while important scopes/features require approval.
- **Unknown:** evidence is insufficient.

For MCP, distinguish an app's own/native MCP server from a third-party integration or Composio connector.

In [ ]:
SYSTEM_PROMPT = """
You are a careful API and integration research analyst.
Use only the supplied evidence. Do not use prior knowledge to fill gaps.
Never invent API endpoints, authentication methods, MCP support, or documentation URLs.
If evidence is insufficient, write Unknown and set needs_manual_review=true.
Prefer official app developer documentation and official support pages.
A search result snippet alone is weak evidence; use page text where available.
Distinguish native MCP support from third-party connectors/integrations.
Return valid JSON only, matching the requested fields.
"""


def extract_app_record(app_name: str, category: str, sources: list) -> dict:
    evidence_text = json.dumps(sources, ensure_ascii=False)
    user_prompt = f"""
Research this application.

App: {app_name}
Category: {category}

Return a JSON object with exactly these fields:
{{
  "app_name": "{app_name}",
  "category": "{category}",
  "description": "One concise sentence describing the product.",
  "authentication_methods": ["OAuth 2.0", "API key", "Basic auth", "JWT", "Other", or "Unknown"],
  "credential_access": "Self-serve, Gated, Mixed, or Unknown",
  "api_available": "Yes, No, or Unknown",
  "api_type_or_surface": "Briefly describe API type/surface, or Unknown",
  "api_docs_url": "Best official documentation URL, or Unknown",
  "mcp_status": "Native MCP, Third-party/Composio integration, No evidence, or Unknown",
  "buildability": "Easy, Moderate, Difficult, or Unknown",
  "main_blocker": "Main practical blocker, or None identified / Unknown",
  "evidence": ["URLs supporting the claims"],
  "confidence": "High, Medium, or Low",
  "needs_manual_review": true,
  "notes": "Important caveats and what is not confirmed."
}}

Rules:
- Do not treat a generic website login as API authentication.
- Explain whether credentials can be obtained self-serve or require approval/sales.
- Only say Native MCP when the app/vendor itself documents an MCP server.
- Third-party connectors do not prove native MCP support.
- Buildability should consider auth setup, API coverage, approval barriers, docs quality,
  rate limits/pricing if shown, and whether useful endpoints exist.
- Evidence must contain URLs from the supplied sources only.
- Set needs_manual_review=true if any important field is Unknown or weakly supported.

Evidence:
{evidence_text}
"""

    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
    )
    raw = response.choices[0].message.content
    data = json.loads(raw)

    # Validate required shape; catches malformed model output.
    validated = AppResearchRecord.model_validate(data)
    return validated.model_dump()


# Test extraction using the sources collected above.
test_record = extract_app_record(
    apps_df.iloc[0]["app_name"],
    apps_df.iloc[0]["category"],
    test_sources
)
display(pd.Series(test_record))

## 9. Run the research agent across the app list

This loop checkpoints after every app. If the notebook stops, rerun it: already completed apps in `research_results.json` will be skipped.

Start with 5–10 apps. Inspect the records. Then run the full 100.

In [ ]:
OUTPUT_JSON = "research_results.json"
OUTPUT_CSV = "research_results.csv"


def load_existing_results(path=OUTPUT_JSON):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_results(records, json_path=OUTPUT_JSON, csv_path=OUTPUT_CSV):
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)
    pd.DataFrame(records).to_csv(csv_path, index=False)


def run_research(apps: pd.DataFrame, limit: Optional[int] = None):
    records = load_existing_results()
    completed = {r.get("app_name") for r in records}

    work = apps.head(limit) if limit else apps

    for _, row in work.iterrows():
        app_name = str(row["app_name"]).strip()
        category = str(row["category"]).strip()

        if app_name in completed:
            print(f"Skipping already completed: {app_name}")
            continue

        print(f"Researching: {app_name}")
        try:
            sources = collect_sources(app_name)
            record = extract_app_record(app_name, category, sources)
            record["source_count"] = len(sources)
            records.append(record)
            save_results(records)
            print(
                f"Saved: {app_name} | API: {record['api_available']} | "
                f"Review: {record['needs_manual_review']}"
            )
        except Exception as e:
            print(f"FAILED: {app_name}: {type(e).__name__}: {e}")
            # Save an explicit failure row so it is not silently lost.
            records.append({
                "app_name": app_name,
                "category": category,
                "description": "Research failed; rerun or review manually.",
                "authentication_methods": [],
                "credential_access": "Unknown",
                "api_available": "Unknown",
                "api_type_or_surface": "Unknown",
                "api_docs_url": "Unknown",
                "mcp_status": "Unknown",
                "buildability": "Unknown",
                "main_blocker": "Research pipeline error",
                "evidence": [],
                "confidence": "Low",
                "needs_manual_review": True,
                "notes": str(e),
                "source_count": 0,
            })
            save_results(records)

        time.sleep(PAUSE_BETWEEN_APPS_SECONDS)

    return records


# FIRST: run only 5 apps and inspect the results.
records = run_research(apps_df, limit=5)
display(pd.DataFrame(records).tail())

## 10. Run all 100 apps

Once the first 5 records look reasonable, change `limit=5` to `limit=None`.

This may take time and incur search/API costs. Keep the notebook open while it runs; checkpoints are saved after each app.

In [ ]:
# Uncomment to run the complete list:
# records = run_research(apps_df, limit=None)
# results_df = pd.DataFrame(records)
# display(results_df)

## 11. Verification agent — independently check a sample

Do not claim a measured accuracy until you compare the agent's results with a manually checked reference. This second pass searches again and asks the model to identify unsupported or conflicting claims.

For a defensible evaluation:
1. Select a random sample (for example, 20 apps).
2. Manually inspect official docs for each sampled app.
3. Record the correct values in a separate `ground_truth.csv`.
4. Compare field by field and calculate accuracy.

The automated second pass is useful for finding likely issues, but it is not a replacement for human verification.

In [ ]:
def verify_app_record(record: dict) -> dict:
    app_name = record["app_name"]
    fresh_results = search_app_docs(
        f"{app_name} official API authentication developer docs MCP"
    )
    fresh_sources = []

    for item in fresh_results[:MAX_PAGES_TO_READ]:
        fresh_sources.append({
            "title": item["title"],
            "url": item["url"],
            "search_snippet": item["content"],
            "page_text": read_webpage(item["url"]),
        })

    prompt = f"""
Review the initial research record against this independently collected evidence.

Initial record:
{json.dumps(record, ensure_ascii=False)}

New evidence:
{json.dumps(fresh_sources, ensure_ascii=False)}

Return JSON with:
{{
  "app_name": "{app_name}",
  "verification_status": "Confirmed, Corrected, Contradicted, or Insufficient evidence",
  "fields_to_change": ["field names"],
  "corrections": {{"field_name": "corrected value"}},
  "reason": "Brief explanation",
  "evidence": ["supporting URLs"],
  "human_review_required": true
}}

Do not assume the initial record is correct. Use Unknown when evidence is insufficient.
"""

    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
    )
    return json.loads(response.choices[0].message.content)


# Verify a small sample first. Increase to 20 after checking costs and output.
results_df = pd.DataFrame(load_existing_results())
sample_size = min(3, len(results_df))
verification_records = []

for _, row in results_df.head(sample_size).iterrows():
    print("Verifying:", row["app_name"])
    verification_records.append(verify_app_record(row.to_dict()))
    time.sleep(1)

with open("verification_results.json", "w", encoding="utf-8") as f:
    json.dump(verification_records, f, indent=2, ensure_ascii=False)

display(pd.DataFrame(verification_records))

## 12. Analyze the findings

This produces descriptive summaries from the collected records. These are observations about your dataset, not universal claims about every app in the market.

In [ ]:
results_df = pd.DataFrame(load_existing_results())

if not results_df.empty:
    print("Apps researched:", results_df["app_name"].nunique())
    print("\nAPI availability:")
    display(results_df["api_available"].value_counts(dropna=False).to_frame("count"))

    print("\nCredential access:")
    display(results_df["credential_access"].value_counts(dropna=False).to_frame("count"))

    print("\nBuildability:")
    display(results_df["buildability"].value_counts(dropna=False).to_frame("count"))

    print("\nMCP status:")
    display(results_df["mcp_status"].value_counts(dropna=False).to_frame("count"))

    print("\nBy category:")
    category_summary = results_df.groupby("category").agg(
        apps=("app_name", "nunique"),
        api_yes=("api_available", lambda s: (s == "Yes").sum()),
        self_serve=("credential_access", lambda s: (s == "Self-serve").sum()),
        needs_review=("needs_manual_review", lambda s: s.astype(str).str.lower().isin(["true", "1"]).sum()),
    ).reset_index()
    display(category_summary)
else:
    print("No research results found yet. Run the research agent first.")

## 13. Find records that need human review

These are the records you should prioritize before final submission.

In [ ]:
if not results_df.empty:
    review_df = results_df[
        results_df["needs_manual_review"].astype(str).str.lower().isin(["true", "1"])
        | results_df["confidence"].isin(["Low", "Medium"])
    ]
    display(review_df[[
        "app_name", "category", "api_available", "credential_access",
        "mcp_status", "confidence", "evidence", "notes"
    ]])
    review_df.to_csv("manual_review_queue.csv", index=False)
    print("Saved manual_review_queue.csv")

## 14. Optional: connect a Composio MCP session

Use this only if your assignment requires you to demonstrate Composio. Composio can expose connected app tools through an MCP session, but that is different from proving that an app provides its own native MCP server.

Install the current Composio SDK and configure `COMPOSIO_API_KEY` in your environment. SDK method names can change between versions, so check the current official quickstart if this cell errors.

In [ ]:
# Optional setup:
# %pip install -q composio

# import os
# from getpass import getpass
# os.environ["COMPOSIO_API_KEY"] = getpass("Composio API key: ").strip()

# Current documented session-style example:
# from composio import Composio
# composio = Composio()
# session = composio.create(user_id="app-research-notebook", mcp=True)
# print("MCP URL:", session.mcp.url)
# print("MCP headers:", session.mcp.headers)

# Do not publish MCP URLs, headers, or credentials in your GitHub repository.

## 15. Export files and download them

The key outputs are:
- `research_results.csv`
- `research_results.json`
- `verification_results.json`
- `manual_review_queue.csv`

Keep your evidence URLs in the final report so a reviewer can reproduce your findings.

In [ ]:
import os

for filename in [
    "research_results.csv",
    "research_results.json",
    "verification_results.json",
    "manual_review_queue.csv",
]:
    if os.path.exists(filename):
        print(filename, "—", os.path.getsize(filename), "bytes")

# In Google Colab only, you can use:
# from google.colab import files
# files.download("research_results.csv")

## Before submitting

- Replace the 10 sample apps with the full 100-app list across all 10 categories.
- Confirm every app is assigned to the correct category.
- Open and inspect the official evidence URLs, especially authentication and credential-access claims.
- Manually verify a sample (for example, 20 apps) and calculate field-level accuracy from your own ground-truth sheet.
- Do not report hypothetical accuracy as measured accuracy.
- Distinguish native MCP support from Composio or other third-party integrations.
- Document blocked pages, unknown fields, API approval barriers, rate limits, pricing, and other buildability blockers.
- Never commit API keys, MCP headers, or other secrets.